In [1]:
import pandas as pd

teens = pd.read_csv("../data/snsdata.csv")
teens.head()

,gradyear,gender,age,friends,basketball,football,soccer,softball,volleyball,swimming,...,blonde,mall,shopping,clothes,hollister,abercrombie,die,death,drunk,drugs
0,2006,M,18.982,7,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,2006,F,18.801,0,0,1,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0
2,2006,M,18.335,69,0,1,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
3,2006,F,18.875,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,2006,NaN,18.995,10,0,0,0,0,0,0,...,0,0,2,0,0,0,0,0,1,1


In [2]:
teens.isna().sum()[lambda s: s > 0]

gender    2724
age       5086
dtype: int64

In [3]:
teens.age.describe()

count    24914.000000
mean        17.993950
std          7.858054
min          3.086000
25%         16.312000
50%         17.287000
75%         18.259000
max        106.927000
Name: age, dtype: float64

In [4]:
import numpy as np

# Ages outside the high-school range are treated as missing
teens.loc[(teens.age < 13) | (teens.age >= 20), "age"] = np.nan

# Missing ages are imputed with the mean age of each graduation year
teens["age"] = teens.groupby("gradyear")["age"].transform(lambda s: s.fillna(s.mean()))
teens.age.describe()

count    30000.000000
mean        17.237326
std          1.141821
min         13.027000
25%         16.282000
50%         17.238000
75%         18.212000
max         19.995000
Name: age, dtype: float64

In [5]:
# Gender dummies, keeping missing gender as its own category
teens["female"] = (teens.gender == "F").astype(int)
teens["no_gender"] = teens.gender.isna().astype(int)
teens[["female", "no_gender"]].sum()

female       22054
no_gender     2724
dtype: int64

In [6]:
from sklearn.preprocessing import StandardScaler

interests = teens.loc[:, "basketball":"drugs"]
interests_scaled = StandardScaler().fit_transform(interests)
interests.shape

(30000, 36)

In [7]:
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans

inertias = []
for k in range(1, 11):
    inertias.append(
        KMeans(n_clusters=k, n_init=10, random_state=42).fit(interests_scaled).inertia_
    )

plt.plot(range(1, 11), inertias, "o-k")
plt.xlabel("Número de clusters")
plt.ylabel("Inercia")
plt.grid(color="lightgray", linestyle="--", linewidth=0.5)
plt.show()

<Figure size 640x480 with 1 Axes>

In [8]:
kmeans = KMeans(n_clusters=5, n_init=10, random_state=42)
teens["cluster"] = kmeans.fit_predict(interests_scaled)
teens.cluster.value_counts().sort_index()

cluster
0      671
1     5412
2     1093
3    22823
4        1
Name: count, dtype: int64

In [9]:
centers = pd.DataFrame(kmeans.cluster_centers_, columns=interests.columns)
centers.round(2).T

,0,1,2,3,4
basketball,0.12,0.49,0.38,-0.14,-0.33
football,0.04,0.51,0.38,-0.14,2.48
soccer,0.02,0.28,0.14,-0.07,-0.24
softball,0.02,0.39,0.17,-0.10,-0.22
volleyball,-0.01,0.43,0.10,-0.11,-0.22
swimming,0.08,0.32,0.27,-0.09,1.67
cheerleading,-0.01,0.44,0.20,-0.11,-0.21
baseball,0.05,0.25,0.29,-0.07,-0.20
tennis,0.11,0.12,0.11,-0.04,-0.17
sports,-0.02,0.30,0.84,-0.11,-0.30


In [10]:
# Top interests per segment
for cluster, row in centers.iterrows():
    print(cluster, ", ".join(row.sort_values(ascending=False).index[:6]))

0 bible, jesus, god, church, death, music
1 shopping, cute, mall, hollister, abercrombie, dance
2 kissed, drugs, hair, sex, drunk, die
3 marching, blonde, tennis, band, death, soccer
4 blonde, sex, drunk, death, hair, die


In [11]:
teens.groupby("cluster")[["age", "female", "friends"]].mean()

,age,female,friends
cluster,,,
0,17.365909,0.752608,35.232489
1,17.005164,0.864560,38.821693
2,17.095725,0.806953,30.589204
3,17.295341,0.700478,27.961355
4,18.119000,1.000000,44.000000


In [12]:
import os

os.makedirs("../submission", exist_ok=True)
teens.to_csv("../submission/segmented.csv", index=True)